# Análise Exploratória de Dados (EDA)
## Análise de Compliance no Setor Público - TCC MBA

**Objetivo:** Explorar os datasets da camada Gold em **nível municipal** (5.570 registros, 1 linha por município brasileiro) para entender:
- Distribuições e estatísticas descritivas dos dados
- Valores ausentes e qualidade dos dados
- Padrões e relações iniciais
- Variações regionais em indicadores de compliance e socioeconômicos

**Nota sobre granularidade.** O principal dataset analítico usado aqui é `analise_compliance_municipio` (uma linha por município, N=5.570). A antiga agregação em nível estadual (`analise_compliance`, N=27) tinha poucas observações para análise de correlação/regressão significativa. Para preservar o contexto geográfico, `codigo_estado` / `nome_estado` / `codigo_regiao` / `nome_regiao` são mantidos como colunas identificadoras e também codificados em variáveis dummy (`is_region_*`, `is_state_*`), do mesmo jeito que o dataset estadual já codificava as regiões.

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## Passo a passo (estilo aula)

1. Pacotes e configuração do ambiente
2. Reprodutibilidade
3. Carregamento dos dados
4. Blocos de análise
5. Resumo e interpretação


# Pacotes


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.analysis.pt_br_loader import GoldDataLoaderPtBr


plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)


In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.useoffset'] = False
mpl.rcParams['axes.formatter.limits'] = (-99, 99)


# Reprodutibilidade


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Semente de reprodutibilidade fixada em {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


## 1. Carregar Dados

In [ ]:
loader = GoldDataLoaderPtBr()

print("Datasets disponíveis:")
for dataset in loader.list_available_datasets():
    print(f"  • {dataset}")


In [ ]:
datasets = loader.load_all()

df_muni = datasets.get('municipio_socioeconomico')
df_state = datasets.get('resumo_estado')
df_sanctions = datasets.get('resumo_sancoes')

# Dataset analítico principal: uma linha por município (N ~= 5.570).
df_analysis = datasets.get('analise_compliance_municipio')

# --- Adiciona dummies one-hot de região e de estado (não estão no dataset muni). ---
# Espelham as colunas is_norte / is_nordeste / ... que existiam no dataset
# estadual -- usando os mesmos nomes humanamente legíveis para que qualquer
# notebook a jusante que referenciasse essas colunas continue funcionando.
# Mantidas como Int64 (0/1) para consistência com a convenção is_* existente
# e para uso direto como features de regressão nos notebooks 02 / 03.
REGION_NAME_TO_DUMMY = {
    'Norte': 'is_norte',
    'Nordeste': 'is_nordeste',
    'Sudeste': 'is_sudeste',
    'Sul': 'is_sul',
    'Centro-Oeste': 'is_centro_oeste',
}
for rname, col in REGION_NAME_TO_DUMMY.items():
    df_analysis[col] = (df_analysis['nome_regiao'] == rname).astype('Int64')
REGION_DUMMY_COLS = list(REGION_NAME_TO_DUMMY.values())

# Dummies de estado: uma por estado, nomeada pelo código IBGE de 2 dígitos.
state_dummies = pd.get_dummies(df_analysis['codigo_estado'], prefix='is_state').astype('Int64')
df_analysis = pd.concat([df_analysis, state_dummies], axis=1)
STATE_DUMMY_COLS = list(state_dummies.columns)

print(f"\n✅ Carregados {len(datasets)} datasets")
print(f"   Dataset analítico principal: analise_compliance_municipio")
print(f"   Linhas: {len(df_analysis):,} municípios, {df_analysis['codigo_estado'].nunique()} estados, {df_analysis['codigo_regiao'].nunique()} regiões")
print(f"   Dummies de região adicionadas ({len(REGION_DUMMY_COLS)}): {REGION_DUMMY_COLS}")
print(f"   Dummies de estado adicionadas ({len(STATE_DUMMY_COLS)}): primeiras 3 = {STATE_DUMMY_COLS[:3]} ... última = {STATE_DUMMY_COLS[-1]}")

## 2. Visão Geral do Dataset

In [ ]:
print("=" * 80)
print("DATASET DE ANÁLISE DE COMPLIANCE (NÍVEL MUNICIPAL)")
print("=" * 80)
print(f"Formato: {df_analysis.shape}  (linhas = municípios, colunas = features + dummies)")
print(f"\nTipos de dados (colunas não-dummy):")
print(df_analysis.drop(columns=REGION_DUMMY_COLS + STATE_DUMMY_COLS).dtypes)
print(f"\nUso de Memória: {df_analysis.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

In [ ]:
# Pré-visualização de algumas colunas (escondendo as 32 dummies de região+estado).
preview_cols = [c for c in df_analysis.columns if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
df_analysis[preview_cols].head(10)

In [ ]:
df_analysis.info(verbose=False)

## 3. Estatísticas Descritivas

In [ ]:
# Descreve apenas as features analíticas, excluindo as 32 dummies one-hot
# (as dummies são resumidas separadamente abaixo).
numeric_cols = df_analysis.select_dtypes(include=[np.number]).columns
analytical_numeric = [c for c in numeric_cols if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
df_analysis[analytical_numeric].describe().T

### Resumo de Métricas Principais

In [ ]:
# Médias ponderadas pela população para taxas. Uma média simples entre 5.570
# municípios trataria uma cidade de 500 habitantes e São Paulo da mesma forma,
# o que é estatisticamente enganoso para indicadores de taxa e valor médio.
_pop = df_analysis['populacao_2022']
_lit_mask = df_analysis['taxa_alfabetizacao_2022'].notna()
_inc_mask = df_analysis['renda_media_2022'].notna()

total_pop = int(_pop.sum())
total_sanc = int(df_analysis['num_sancoes'].sum())

summary = pd.DataFrame({
    'Total de Municípios': [len(df_analysis)],
    'Total de Estados': [int(df_analysis['codigo_estado'].nunique())],
    'Total de Regiões': [int(df_analysis['codigo_regiao'].nunique())],
    'População Total (2022)': [total_pop],
    'Total de Sanções': [total_sanc],
    'Sanções/100k (nacional, ponderada)': [round(total_sanc / total_pop * 100_000, 2)],
    'Alfabetização % (ponderada)': [round(np.average(df_analysis.loc[_lit_mask, 'taxa_alfabetizacao_2022'], weights=_pop[_lit_mask]), 2)],
    'Renda Média BRL (ponderada)': [round(np.average(df_analysis.loc[_inc_mask, 'renda_media_2022'], weights=_pop[_inc_mask]), 2)],
})

summary.T

**Resumo das variáveis dummy.** Para as dummies one-hot de região e estado, a média equivale à proporção de municípios naquela região / estado (ex.: `df['is_region_1'].mean()` é a fração de municípios na região Norte). Um resumo curto é mostrado abaixo para não poluir a tabela principal do `.describe()` com 32 linhas extras.

In [ ]:
dummy_stats = pd.DataFrame({
    'Contagem (=1)': df_analysis[REGION_DUMMY_COLS + STATE_DUMMY_COLS].sum(),
    'Fração de municípios %': (df_analysis[REGION_DUMMY_COLS + STATE_DUMMY_COLS].mean() * 100).round(2),
}).sort_values('Contagem (=1)', ascending=False)
print(f"Total de dummies: {len(dummy_stats)} (5 regiões + {len(STATE_DUMMY_COLS)} estados)")
dummy_stats.head(10)

## 4. Análise de Valores Ausentes

In [ ]:
missing = df_analysis.isnull().sum()
missing_pct = (missing / len(df_analysis)) * 100

missing_df = pd.DataFrame({
    'Qtde Ausente': missing,
    'Ausente %': missing_pct.round(2),
}).sort_values('Qtde Ausente', ascending=False)

missing_df[missing_df['Qtde Ausente'] > 0]

## 5. Análise de Distribuição

### 5.1 Variável Alvo: Sanções por 100 mil

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_analysis['sancoes_por_100k'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribuição de Sanções por 100 mil', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sanções por 100 mil habitantes')
axes[0].set_ylabel('Frequência')
axes[0].axvline(df_analysis['sancoes_por_100k'].mean(), color='red', linestyle='--', label='Média')
axes[0].axvline(df_analysis['sancoes_por_100k'].median(), color='green', linestyle='--', label='Mediana')
axes[0].legend()

axes[1].boxplot(df_analysis['sancoes_por_100k'])
axes[1].set_title('Boxplot: Sanções por 100 mil', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sanções por 100 mil habitantes')

from scipy import stats
stats.probplot(df_analysis['sancoes_por_100k'], dist="norm", plot=axes[2])
axes[2].set_title('Gráfico Q-Q: Sanções por 100 mil', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Assimetria: {df_analysis['sancoes_por_100k'].skew():.3f}")
print(f"Curtose: {df_analysis['sancoes_por_100k'].kurtosis():.3f}")


### 5.2 Indicadores Socioeconômicos

**Nota sobre a Transformação Logarítmica:** Dados populacionais são tipicamente muito assimétricos à direita — poucos estados têm populações extremamente grandes, enquanto a maioria tem valores muito menores. A transformação logarítmica (log_populacao) resolve isso ao:
1. **Normalizar a distribuição** — tornando-a mais simétrica para análises estatísticas válidas
2. **Reduzir a influência de valores extremos** — evitando que populações grandes afetem desproporcionalmente correlações e regressões
3. **Permitir interpretação percentual** — em modelos de regressão, mudanças representam efeitos proporcionais

Os histogramas abaixo comparam a distribuição populacional bruta (assimétrica) com a versão transformada em log (mais normal).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].hist(df_analysis['taxa_alfabetizacao_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].set_title('Distribuição da Taxa de Alfabetização 2022 (por município)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Taxa de Alfabetização (%)')
axes[0, 0].set_ylabel('Frequência')

axes[0, 1].hist(df_analysis['renda_media_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 1].set_title('Distribuição da Renda Média 2022 (por município)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Renda Média (BRL)')
axes[0, 1].set_ylabel('Frequência')

axes[1, 0].hist(df_analysis['populacao_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='salmon')
axes[1, 0].set_title('Distribuição da População 2022 (por município)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('População')
axes[1, 0].set_ylabel('Frequência')

axes[1, 1].hist(df_analysis['log_populacao'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='plum')
axes[1, 1].set_title('Distribuição do Log(População) (por município)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Log(População)')
axes[1, 1].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

## 6. Análise Regional

In [ ]:
# Agregação regional a partir de dados em NÍVEL MUNICIPAL, usando
# estatísticas PONDERADAS PELA POPULAÇÃO. Um groupby('nome_regiao').agg('mean')
# simples trataria cada município igualmente, o que é estatisticamente
# enganoso: cidades pequenas dominariam a média para taxas e valores monetários.
# Para taxas e médias regionais, agregamos a partir dos totais e ponderamos
# pela população municipal.

def _region_rollup(g: pd.DataFrame) -> pd.Series:
    pop = g['populacao_2022']
    lit_mask = g['taxa_alfabetizacao_2022'].notna()
    inc_mask = g['renda_media_2022'].notna()
    return pd.Series({
        'N Municípios': len(g),
        'N Estados': g['codigo_estado'].nunique(),
        'População Total': int(pop.sum()),
        'Total de Sanções': int(g['num_sancoes'].sum()),
        'Sanções/100k (ponderada)': round(g['num_sancoes'].sum() / pop.sum() * 100_000, 2),
        'Alfabetização % (ponderada)': round(np.average(g.loc[lit_mask, 'taxa_alfabetizacao_2022'], weights=pop[lit_mask]), 2),
        'Renda BRL (ponderada)': round(np.average(g.loc[inc_mask, 'renda_media_2022'], weights=pop[inc_mask]), 2),
    })

regional_summary = (
    df_analysis.groupby('nome_regiao', observed=True)
    .apply(_region_rollup)
)

regional_summary

In [ ]:
# Painel regional construído a partir dos dados municipais (taxas ponderadas pela população).
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sanções por 100 mil por Região (ponderada)',
                    'Taxa de Alfabetização por Região (ponderada)',
                    'Renda Média por Região (ponderada)',
                    'Total de Sanções por Região'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

regions = regional_summary.reset_index()

fig.add_trace(go.Bar(x=regions['nome_regiao'], y=regions['Sanções/100k (ponderada)'],
                     name='Sanções/100k', marker_color='indianred'), row=1, col=1)
fig.add_trace(go.Bar(x=regions['nome_regiao'], y=regions['Alfabetização % (ponderada)'],
                     name='Alfabetização %', marker_color='lightseagreen'), row=1, col=2)
fig.add_trace(go.Bar(x=regions['nome_regiao'], y=regions['Renda BRL (ponderada)'],
                     name='Renda', marker_color='lightsalmon'), row=2, col=1)
fig.add_trace(go.Bar(x=regions['nome_regiao'], y=regions['Total de Sanções'],
                     name='Total de Sanções', marker_color='mediumpurple'), row=2, col=2)

fig.update_layout(height=800, showlegend=False, title_text="Painel Regional (construído a partir de 5.570 municípios)")
fig.show()

## 7. Agregação por Estado e Extremos Municipais

Analisamos os dois extremos do espectro de granularidade:

1. **Agregação por estado** (ponderada pela população, a partir dos 5.570 municípios) para um gráfico de barras estável e relevante para políticas públicas.
2. **Top / bottom municípios** por `sancoes_por_100k`, capaz de revelar outliers individuais que a agregação por estado esconde.

In [ ]:
# Top 10 e bottom 10 MUNICÍPIOS por sanções por 100 mil hab.
# OBS: taxas per capita para municípios muito pequenos podem ser instáveis
# (efeito denominador) -- usar com cautela.
cols = ['codigo_municipio', 'nome_municipio', 'nome_estado', 'nome_regiao',
        'populacao_2022', 'num_sancoes', 'sancoes_por_100k']

top_10_munis = df_analysis.nlargest(10, 'sancoes_por_100k')[cols]

print("TOP 10 MUNICÍPIOS - Maiores Sanções por 100 mil")
print("=" * 90)
print(top_10_munis.to_string(index=False))

print("\n\nBOTTOM 10 MUNICÍPIOS - Menores Sanções por 100 mil (entre os com sanções > 0)")
print("=" * 90)
with_sanctions = df_analysis[df_analysis['num_sancoes'] > 0]
print(with_sanctions.nsmallest(10, 'sancoes_por_100k')[cols].to_string(index=False))

In [ ]:
# Agregação por estado a partir dos dados municipais (ponderada pela população).
# nome_estado é usado no eixo x (rótulo qualitativo); codigo_estado é apenas um id.
state_rollup = (
    df_analysis.groupby(['codigo_estado', 'nome_estado', 'nome_regiao'], observed=True)
    .apply(lambda g: pd.Series({
        'populacao': g['populacao_2022'].sum(),
        'num_sancoes': g['num_sancoes'].sum(),
        'sancoes_por_100k': g['num_sancoes'].sum() / g['populacao_2022'].sum() * 100_000,
    }))
    .reset_index()
    .sort_values('sancoes_por_100k', ascending=False)
)

fig = px.bar(state_rollup,
             x='nome_estado', y='sancoes_por_100k',
             color='nome_regiao',
             title='Sanções por 100 mil habitantes por Estado (agregado a partir de 5.570 municípios)',
             labels={'sancoes_por_100k': 'Sanções por 100 mil', 'nome_estado': 'Estado'},
             height=500)
fig.update_xaxes(tickangle=-45)
fig.show()

## 8. Análise de Registros de Sanções

In [ ]:
if df_sanctions is not None:
    print("Sanções por Tipo de Registro:")
    print("=" * 60)
    display(df_sanctions[['tipo_registro', 'total_sancoes', 'sancoes_pf', 'sancoes_pj', 'razao_pj_pct']])
    
    fig = px.pie(df_sanctions, values='total_sancoes', names='tipo_registro',
                 title='Distribuição de Sanções por Tipo de Registro')
    fig.show()


## 9. Mapa de Calor de Correlação (Prévia)

In [ ]:
# Matriz de correlação das principais features analíticas em nível municipal
# (5.570 observações em vez de 27 estados -- muito mais poder estatístico).
# Incluímos as dummies de região, mas NÃO as 27 dummies de estado (deixariam
# o heatmap ilegível). Features do lado das transferências também são
# incluídas, já que a pergunta do TCC liga transferências federais a
# resultados de compliance.

corr_cols = ['sancoes_por_100k', 'taxa_alfabetizacao_2022', 'renda_media_2022',
             'log_populacao', 'log_renda']
if 'log_total_transferencias' in df_analysis.columns:
    corr_cols.append('log_total_transferencias')
corr_cols += REGION_DUMMY_COLS

# Cast para float64 (numpy) para np.corrcoef / seaborn aceitarem tipos
# Int64/Float64 nullable do pandas junto com linhas que contenham NaN.
corr_df = df_analysis[corr_cols].astype('Float64').astype(float)
corr_matrix = corr_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação: Variáveis Principais (nível municipal, N=5.570)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Resumo das Principais Descobertas

### 10.1 Qualidade dos Dados
- **Granularidade:** esta EDA roda em **nível municipal** (5.570 linhas, 1 por município brasileiro), ao invés da antiga visão estadual (27 linhas). Isso dá ~200x mais poder estatístico para o trabalho de correlação e regressão a jusante.
- **Cobertura geográfica completa**: todos os 27 estados e todas as 5 regiões estão representados.
- **Dados de sanções são densos**: `num_sancoes` é não-nulo para todos os 5.570 municípios (zeros são reais, não ausentes).
- **Feature de taxa de sanções por transferência é esparsa**: `sancoes_por_milhao_brl_transferencias` é nulo para ~91,5% dos municípios (a maioria não tem registros de transferência federal no corte Gold atual). Usar com cautela em qualquer modelo que dependa dela.

### 10.2 Padrões Regionais (ponderados pela população)
- Os números regionais são calculados como `sum(sancoes) / sum(populacao) * 100_000` entre os municípios de cada região -- não como média simples das taxas municipais -- então não são dominados por municípios pequenos.
- Depois da ponderação correta, a ordenação das regiões por sanções/100k é mais achatada do que a antiga visão estadual sem ponderação sugeria; ver a tabela de resumo regional acima para os valores exatos no snapshot Gold atual.

### 10.3 Extremos Estaduais e Municipais
- O gráfico de barras por estado agora é calculado como uma **agregação a partir dos dados municipais** (ponderada pela população), então o número de cada estado fica consistente com os totais regionais.
- Os top-10 municípios por `sancoes_por_100k` revelam outliers individuais que a agregação estadual esconde. Taxas de municípios muito pequenos podem ser instáveis (efeito denominador) e devem ser interpretadas junto com `num_sancoes` absoluto e `populacao_2022`.

### 10.4 Variáveis dummy
- Região e estado são preservados como colunas identificadoras (`codigo_estado`, `nome_estado`, `codigo_regiao`, `nome_regiao`) E como dummies one-hot (`is_region_*`, `is_state_*`) para uso como features de regressão.
- A média de uma dummy equivale à proporção de municípios naquela categoria (ex.: `df['is_region_1'].mean()` == fração de municípios na região Norte). Ver a tabela de resumo das dummies para as frações exatas.

### 10.5 Correlações (nível municipal)
- Com N=5.570 os coeficientes de correlação no heatmap são muito mais confiáveis do que no grão estadual de 27 linhas. Inspecionar a linha / coluna de `sancoes_por_100k` no heatmap acima para os sinais bivariados mais fortes; a pergunta do TCC sobre transferências federais vs. resultados de compliance agora pode ser testada na unidade de observação onde a política realmente aterriza (o município).